In [ ]:
meta = {
    'titulo': 'RetailNova Case 1 · Dataset sintético validado',
    'num_registros': int(len(final_df)),
    'fecha_generacion': pd.Timestamp.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC'),
    'variables_clave': ['region', 'segmento', 'plan', 'churn_6m', 'propensity_to_buy', 'incidencias_tecnicas', 'latencia_ms'],
    'prevalencia_churn': round(float(final_df['churn_6m'].mean()), 4),
    'rango_edad': [int(final_df['edad'].min()), int(final_df['edad'].max())],
    'reglas_documentadas': REGLAS_DOCUMENTADAS,
    'restricciones': restricciones,
    'ruta_csv': str(csv_path),
}
json_path = OUT_DIR / 'retailnova_case1_sintetico_metadata.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('Metadata guardada en:', json_path)
print(json.dumps({'filas': len(final_df), 'churn': round(float(final_df['churn_6m'].mean()), 4)}, ensure_ascii=False, indent=2))

## 12. Guardado y documentación del dataset generado

La notebook queda preparada para usar el dataset generado en validación de sesgos, EDA y modelado predictivo. Además, se guardan metadatos que permiten reproducir la lógica del caso 1 y documentar las reglas aplicadas.

In [ ]:
# Exportación y particiones base
final_cols = [
    'id_cliente','fecha_alta','edad','genero','region','zona','segmento','canal_venta','tipo_contrato','plan',
    'ingresos_mensuales','antiguedad_meses','ticket_promedio','uso_datos_gb','minutos_mes','llamadas_perdidas',
    'incidencias_tecnicas','sla','nps','riesgo_crediticio','prioridad_atencion','churn_6m','propensity_to_buy',
    'migracion','mora','calidad_experiencia','latencia_ms','throughput_mbps','tasa_fallos','congestion_red'
]

final_df = base[final_cols].copy()
final_df = final_df.sort_values('id_cliente').reset_index(drop=True)

csv_path = OUT_DIR / 'retailnova_case1_sintetico.csv'
final_df.to_csv(csv_path, index=False)
print('Dataset exportado en:', csv_path)
print(final_df.shape)
final_df.head()

## 11. Construcción del dataset final con particiones y archivos exportables

El dataset final se consolida en una tabla única con columnas bien nombradas, tipos consistentes y una proporción de churn que refleja el caso de negocio. Se exporta como CSV para uso en modelos y análisis exploratorio.

In [ ]:
assert base['id_cliente'].is_unique
assert base['edad'].between(18, 75).all()
assert base['antiguedad_meses'].between(1, 96).all()
assert base['nps'].between(0, 100).all()
assert base['churn_6m'].isin([0, 1]).all()
assert base['migracion'].isin([0, 1]).all()
assert base['mora'].isin([0, 1]).all()
assert base['p_churn'].between(0, 1).all()
assert (base['region'].value_counts(normalize=True).reindex(region_order).fillna(0) > 0).all()
print('Regiones:', base['region'].value_counts(normalize=True).sort_index().to_dict())
print('Segmentos:', base['segmento'].value_counts(normalize=True).sort_index().to_dict())
print('Churn global:', round(base['churn_6m'].mean(), 4))
print('Churn por región:', round(base.groupby('region')['churn_6m'].mean().sort_values(ascending=False), 4).to_dict())
print('Churn por segmento:', round(base.groupby('segmento')['churn_6m'].mean().sort_values(ascending=False), 4).to_dict())

## 10. Validación de integridad, consistencia y cumplimiento de restricciones

Se aplica un conjunto de validaciones de negocio y de dato: rangos, columnas obligatorias, partición válida, proporciones de regiones y segmentación, y coherencia entre riesgo y churn. Esto ayuda a que el dataset cumpla con la intención del caso y deje una base reproducible para modelado.

In [ ]:
# Ajuste final de etiquetas y escenarios de negocio
base['propensity_to_buy'] = np.clip(base['propensity_to_buy'] * 1.2 + 0.10 * (base['migracion'] == 1), 0, 1).round(3)
base['prioridad_atencion'] = np.where(base['churn_6m'] == 1, 'Alta', np.where(base['incidencias_tecnicas'] > 4, 'Media', 'Baja'))
base['migracion'] = np.where(base['propensity_to_buy'] > 0.8, rng.integers(0, 2, size=n), 0).astype(int)
base['mora'] = np.where(base['mora'] == 1, 1, np.where(base['ingresos_mensuales'] < 800000, rng.integers(0, 2, size=n), 0)).astype(int)
base['calidad_experiencia'] = np.select([
    base['nps'] <= 25,
    base['nps'] <= 60,
    base['nps'] > 60,
], ['Mala', 'Regular', 'Buena'], default='Buena')
base[['churn_6m', 'propensity_to_buy', 'migracion', 'mora', 'prioridad_atencion', 'calidad_experiencia']].head()

## 9. Generación de etiquetas y escenarios de negocio

Se crean etiquetas de churn, intención de compra, migración y prioridad de atención. El caso de negocio se apoya en la lógica de priorizar revisión de clientes con mayor riesgo y peor experiencia, sin automatizar campañas ni asumir causalidad.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

base['p_churn'] = sigmoid(base['churn_score_raw'] + 0.25 * (base['region'] == 'Sur').astype(int) + 0.10 * (base['segmento'] == 'Pyme').astype(int))
base['churn_6m'] = (base['p_churn'] > 0.5).astype(int)
base['propensity_to_buy'] = np.clip(base['propensity_to_buy'] - 0.15 * base['churn_6m'] + 0.10 * (base['nps'] >= 70), 0, 1)
base['prioridad_atencion'] = np.where(base['churn_6m'] == 1, 'Alta', base['prioridad_atencion'])
base['calidad_experiencia'] = np.where(base['churn_6m'] == 1, 'Mala', base['calidad_experiencia'])
print('Prevalencia churn estimada:', round(base['churn_6m'].mean(), 4))
base[['region', 'segmento', 'plan', 'churn_6m', 'p_churn', 'prioridad_atencion']].head()

## 8. Modelado de sesgos estructurales y reglas de equidad

Se integran sesgos explícitos pero controlados para reflejar la realidad de negocio: más incidencias en zonas con peor red, más churn en segmentos de bajo valor o con peor servicio, y diferencias de acceso a planes premium. El objetivo no es justificar discriminación, sino construir un caso realista para análisis de riesgo y validación de fairness.

In [ ]:
base['latencia_ms'] = np.clip(rng.normal(42, 20, size=n) + (base['region'] == 'Sur').astype(int) * 14 + (base['plan'] == 'Básico').astype(int) * 10, 10, 250).round(1)
base['throughput_mbps'] = np.clip(rng.normal(75, 22, size=n) - (base['plan'] == 'Básico').astype(int) * 18 + (base['segmento'] == 'Corporativo').astype(int) * 30, 5, 250).round(1)
base['tasa_fallos'] = np.clip(rng.beta(2.5, 7, size=n) + (base['region'] == 'Sur').astype(float) * 0.12 + (base['plan'] == 'Básico').astype(float) * 0.10, 0, 0.9).round(4)
base['congestion_red'] = np.clip(rng.normal(18, 11, size=n) + base['incidencias_tecnicas'] * 1.6, 0, 95).round(1)
base['migracion'] = np.where(base['propensity_to_buy'] > 0.72, rng.integers(0, 2, size=n), 0).astype(int)
base['mora'] = np.where(base['riesgo_crediticio'] == 'Alto', rng.integers(0, 2, size=n), 0).astype(int)
base['churn_score_raw'] = (
    -1.5
    + 0.45 * (base['region'] == 'Sur').astype(int)
    + 0.30 * (base['segmento'] == 'Retail').astype(int)
    + 0.25 * (base['plan'] == 'Básico').astype(int)
    + 0.32 * (base['canal_venta'] == 'Call center').astype(int)
    + 0.22 * (base['riesgo_crediticio'] == 'Alto').astype(int)
    + 0.26 * (base['calidad_experiencia'] == 'Mala').astype(int)
    + 0.15 * (base['nps'] < 40).astype(int)
    + 0.12 * (base['incidencias_tecnicas'] > 3).astype(int)
    + 0.10 * (base['latencia_ms'] > 80).astype(int)
    + 0.05 * (base['tasa_fallos'] > 0.15).astype(int)
)
base[['latencia_ms', 'throughput_mbps', 'tasa_fallos', 'congestion_red', 'migracion', 'mora', 'churn_score_raw']].head()

## 7. Incorporación de métricas técnicas y variables de telecom

Aquí se agregan indicadores de calidad de servicio y redes, con estructura que permite espejo del caso telco y del caso de churn. Esto facilita la validación de sesgos por región, tipo de línea y experiencia del usuario.

## 6. Simulación de comportamiento operativo y uso del servicio

Se incorporan variables de consumo operativo: datos usados, minutos, anomalías de red, llamadas perdidas, fallas técnicas y experiencia. Estas variables permiten modelar churn de forma diferenciada por tipo de servicio y nivel de soporte.

In [ ]:
# Sesgos estructurales por región y segmento
base['region_score'] = base['region'].map({'Norte': 0.9, 'Centro': 0.75, 'Sur': 0.58, 'Metropolitano': 1.1})
base['segment_score'] = base['segmento'].map({'Retail': 1.0, 'Pyme': 0.95, 'Corporativo': 0.82, 'Gobierno': 0.74})
base['plan_score'] = base['plan'].map({'Básico': 1.18, 'Estandar': 0.96, 'Plus': 0.82, 'Premium': 0.70})
base['canal_score'] = base['canal_venta'].map({'Digital': 1.02, 'Agencia': 0.94, 'Call center': 0.88, 'Partner': 0.86})
base['risk_signal'] = (
    (base['incidencias_tecnicas'] * 0.12)
    + (base['llamadas_perdidas'] * 0.06)
    + (base['nps'] < 40).astype(int) * 0.60
    + (base['riesgo_crediticio'] == 'Alto').astype(int) * 0.40
    + (base['plan'] == 'Básico').astype(int) * 0.28
    + (base['segmento'] == 'Retail').astype(int) * 0.22
    - (base['segmento'] == 'Corporativo').astype(int) * 0.10
)
base['risk_signal'] = base['risk_signal'].clip(0, 4)
base[['region_score', 'segment_score', 'plan_score', 'canal_score', 'risk_signal']].head()

## 5. Generación de variables demográficas y segmentación

Se distribuyen clientes por región, segmento y canal con sesgos sintéticos controlados, de modo que el dataset resulte plausible para análisis de negocio y, al mismo tiempo, capture diferencias sistemáticas en calidad de servicio, riesgo de churn y capacidad de compra.

In [ ]:
# Generación de comportamiento operativo y consumo
base['uso_datos_gb'] = np.clip(rng.normal(48, 20, size=n) + (base['segmento'].map({'Retail': 4, 'Pyme': 10, 'Corporativo': 26, 'Gobierno': 18}) + base['edad'] / 10), 1, 300).round(2)
base['minutos_mes'] = np.clip(rng.normal(240, 120, size=n) + base['segmento'].map({'Retail': 30, 'Pyme': 65, 'Corporativo': 110, 'Gobierno': 85}), 10, 2500).round(2)
base['llamadas_perdidas'] = np.clip(rng.poisson(1.8, size=n) + (base['segmento'] == 'Retail').astype(int) * 2, 0, 30).astype(int)
base['incidencias_tecnicas'] = np.clip(rng.poisson(1.2, size=n) + (base['region'].isin(['Sur','Norte'])).astype(int) * 2 + (base['plan'] == 'Básico').astype(int) * 2, 0, 30).astype(int)
base['propensity_to_buy'] = np.clip(rng.uniform(0.05, 0.95, size=n) + (base['nps'] > 70).astype(int) * 0.2 - (base['incidencias_tecnicas'] > 3).astype(int) * 0.15, 0, 1).round(3)
base['mora'] = np.where(base['riesgo_crediticio'] == 'Alto', rng.integers(0, 2, size=n), rng.integers(0, 1, size=n)).astype(int)
base[['uso_datos_gb', 'minutos_mes', 'llamadas_perdidas', 'incidencias_tecnicas', 'propensity_to_buy', 'mora']].head()

In [ ]:
# Variables demográficas y segmentación
segment_map = {
    'Retail': {'ingreso_factor': 0.82, 'age_shift': -3, 'ticket_base': 140000},
    'Pyme': {'ingreso_factor': 1.05, 'age_shift': 0, 'ticket_base': 260000},
    'Corporativo': {'ingreso_factor': 1.33, 'age_shift': 4, 'ticket_base': 520000},
    'Gobierno': {'ingreso_factor': 1.20, 'age_shift': 6, 'ticket_base': 450000},
}

base['ticket_promedio'] = base.apply(
    lambda r: r['ingresos_mensuales'] * (0.18 + rng.random() * 0.10) + segment_map[r['segmento']]['ticket_base'],
    axis=1
)
base['nps'] = np.clip(rng.normal(64, 17, size=n), 0, 100).round().astype(int)
base['riesgo_crediticio'] = np.select(
    [base['ingresos_mensuales'] < 850000, base['ingresos_mensuales'] < 1_600_000, base['ingresos_mensuales'] >= 1_600_000],
    ['Alto', 'Medio', 'Bajo'],
    default='Medio'
)
base['prioridad_atencion'] = np.where(base['nps'] < 40, 'Alta', np.where(base['riesgo_crediticio'] == 'Alto', 'Media', 'Baja'))
base['sla'] = np.where(base['segmento'].isin(['Corporativo', 'Gobierno']), 'Platinum', np.where(base['segmento'] == 'Pyme', 'Gold', 'Standard'))
base['calidad_experiencia'] = np.select(
    [base['nps'] < 35, base['nps'] < 60, base['nps'] >= 60],
    ['Mala', 'Regular', 'Buena'],
    default='Buena'
)
base['ingresos_mensuales'] = base['ingresos_mensuales'].round(2)
base[['segmento', 'canal_venta', 'plan', 'riesgo_crediticio', 'sla', 'nps', 'prioridad_atencion']].head()

## 4. Diseño de entidades y población base

Se genera una población de clientes con estructura suficiente para reflejar la heterogeneidad del negocio: ingreso relativo, segmentación comercial, canal de adquisición, tipo de contrato y región. Las tasas de churn y la peor calidad de servicio se modelan como variables dependientes del perfil, no como ruido aleatorio.

In [ ]:
n = 150000
rng = np.random.default_rng(SEED)

region_order = ['Norte', 'Centro', 'Sur', 'Metropolitano']
segment_order = ['Retail', 'Pyme', 'Corporativo', 'Gobierno']
plan_order = ['Básico', 'Estandar', 'Plus', 'Premium']
canal_order = ['Digital', 'Agencia', 'Call center', 'Partner']
contract_order = ['Prepago', 'Móvil', 'Fijo', 'Combo']

region_probs = np.array([0.09, 0.26, 0.18, 0.47])
segment_probs = np.array([0.55, 0.28, 0.12, 0.05])

base = pd.DataFrame({
    'id_cliente': np.arange(1, n + 1),
    'fecha_alta': pd.to_datetime('2020-01-01') + pd.to_timedelta(rng.integers(0, 2200, size=n), unit='D'),
    'region': rng.choice(region_order, size=n, p=region_probs),
    'segmento': rng.choice(segment_order, size=n, p=segment_probs),
    'canal_venta': rng.choice(canal_order, size=n, p=[0.44, 0.30, 0.16, 0.10]),
    'tipo_contrato': rng.choice(contract_order, size=n, p=[0.30, 0.27, 0.18, 0.25]),
    'plan': rng.choice(plan_order, size=n, p=[0.36, 0.35, 0.20, 0.09]),
    'genero': rng.choice(['Femenino', 'Masculino', 'No binario'], size=n, p=[0.49, 0.48, 0.03]),
    'edad': np.clip(rng.normal(39, 15, size=n), 18, 75).astype(int),
    'ingresos_mensuales': np.clip(rng.lognormal(mean=4.0, sigma=0.55, size=n) * 220000, 400000, 3_500_000),
    'antiguedad_meses': np.clip(rng.integers(1, 96, size=n), 1, 95),
})

base['zona'] = np.where(base['region'].isin(['Sur', 'Norte']), 'Rural/Intermedia', 'Urbana/Metropolitana')
base[['id_cliente', 'region', 'segmento', 'plan', 'canal_venta', 'edad', 'ingresos_mensuales']].head()

In [ ]:
docs_files = sorted((ROOT / 'docs').glob('*')) if (ROOT / 'docs').exists() else []
notebooks_files = sorted((ROOT / 'notebooks').glob('*')) if (ROOT / 'notebooks').exists() else []
print('Archivos de docs:', [p.name for p in docs_files])
print('Archivos de notebooks:', [p.name for p in notebooks_files])

for p in docs_files[:5]:
    print('-', p.name, p.suffix)

# lista de restricciones que se convertirán en reglas programáticas
restricciones = [
    'Cada fila representa a un cliente o un punto de servicio para un corte temporal.',
    'La prevalencia de churn debe ser alta pero razonable para modelado y priorización.',
    'Region, segmento, canal y plan determinan la probabilidad de churn y la calidad observada.',
    'Hay sesgo estructural por zona geográfica y estrategia comercial.',
    'La base no puede presentar duplicados ni identificadores no únicos.',
    'Debe existir una etiqueta binaria de churn y variables de prioridad de atención.',
]
print('\nRestricciones del caso 1:')
for idx, regla in enumerate(restricciones, start=1):
    print(f'{idx}. {regla}')

## 3. Análisis de documentos del repositorio y extracción de reglas del caso 1

La lectura de la documentación y la experiencia de los proyectos de referencia muestran que el dataset debe ser coherente desde dos perspectivas:
1. realista para negocio y operación;
2. controlado para modelado predictivo y auditoría de sesgos.

Para ello se incorporan reglas estructurales por región, segmento, canal, tipo de contrato y servicio, evitando generar un conjunto “perfectamente equilibrado” que no refleje las desigualdades sistémicas que el caso quiere evaluar.

In [ ]:
REGLAS_DOCUMENTADAS = {
    'objetivo_negocio': 'Priorizar clientes con mayor riesgo de churn operativo para atención comercial y soporte técnico.',
    'metricas_prioridad': 'No basta accuracy; debe priorizarse recall, sensibilidad por churn y control de falsos negativos en segmentos críticos.',
    'train_test_contract': 'El split debe proteger la fuga: todo preprocesamiento y selección se aprenden solo sobre train.',
    'sesgo_estructural': 'Debe haber diferencias por región, segmento, antigüedad, canal y plan que reflejen condiciones reales del mercado y no meras anomalías aleatorias.',
    'calidad_datos': 'Se deben respetar tipos, rangos, nulos y relaciones lógicas; no puede haber filas inconsistentes ni variables imposibles.',
    'no_causalidad': 'El dataset sintetizado no pretende probar causalidad, solo debe ser coherente con escenarios de negocio realistas.',
    'target_binary': 'churn_6m debe mantenerse binario y con prevalencia razonable para modelado supervisado.',
    'alerta_equidad': 'Los sectores y zonas con peor servicio pueden presentar mayor tasa de churn y más incidencias de soporte.',
}

pd.Series(REGLAS_DOCUMENTADAS).to_frame('regla')

## 2. Definición del esquema del dataset y variables objetivo

El dataset final debe contener un registro por cliente con al menos una vista histórica y una etiqueta de comportamiento. Se definen variables de negocio, de riesgo, de uso y de calidad para soportar modelado predictivo, diagnóstico de sesgos y validación de cumplimiento del caso 1.

In [ ]:
SCHEMA = [
    ('id_cliente', 'int'),
    ('fecha_alta', 'datetime'),
    ('edad', 'int'),
    ('genero', 'category'),
    ('region', 'category'),
    ('zona', 'category'),
    ('segmento', 'category'),
    ('canal_venta', 'category'),
    ('tipo_contrato', 'category'),
    ('plan', 'category'),
    ('ingresos_mensuales', 'float'),
    ('antiguedad_meses', 'int'),
    ('ticket_promedio', 'float'),
    ('uso_datos_gb', 'float'),
    ('minutos_mes', 'float'),
    ('llamadas_perdidas', 'int'),
    ('incidencias_tecnicas', 'int'),
    ('sla', 'category'),
    ('nps', 'int'),
    ('riesgo_crediticio', 'category'),
    ('prioridad_atencion', 'category'),
    ('churn_6m', 'int'),
    ('propensity_to_buy', 'float'),
    ('migracion', 'int'),
    ('mora', 'int'),
    ('calidad_experiencia', 'category')
]

SCHEMA_DF = pd.DataFrame(SCHEMA, columns=['columna', 'tipo'])
SCHEMA_DF

## 1. Configuración del entorno y carga de requisitos

Se preparan las bibliotecas y las reglas de negocio que guiarán la disparidad estructural del caso 1. El dataset sintético no debe ser aleatorio: debe respetar rangos, distribuciones y sesgos condicionados que reflejen negocio y riesgo operativos.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

ROOT = Path.cwd().resolve()
DOCS_DIR = ROOT / 'docs'
NOTEBOOKS_DIR = ROOT / 'notebooks'
OUT_DIR = ROOT / 'data' / 'generated'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RULES = {
    'min_clientes': 120000,
    'target_ratio': 0.22,
    'region_bias': {'Norte': 0.09, 'Centro': 0.26, 'Sur': 0.18, 'Metropolitano': 0.47},
    'segment_bias': {'Retail': 0.55, 'Pyme': 0.28, 'Corporativo': 0.12, 'Gobierno': 0.05},
    'churn_by_segment': {'Retail': 0.30, 'Pyme': 0.22, 'Corporativo': 0.09, 'Gobierno': 0.05},
    'premium_penalty': {'Bajo': 0.18, 'Medio': 0.12, 'Alto': 0.06},
    'canal_bias': {'Digital': 0.44, 'Agencia': 0.30, 'Call center': 0.16, 'Partner': 0.10},
    'service_time_bias': {'Norte': 1.15, 'Centro': 1.0, 'Sur': 0.88, 'Metropolitano': 1.12},
    'min_age': 18,
    'max_age': 75,
    'min_antiguedad': 1,
    'max_antiguedad': 144,
    'max_ticket': 2_500_000,
    'max_meses': 96,
}

print(f'Raíz de trabajo: {ROOT}')
print(f'Carpetas docs/notebooks: {DOCS_DIR.exists()} | {NOTEBOOKS_DIR.exists()}')

# RetailNova · Generador de dataset sintético para el Caso 1

Notebook orientada a crear un dataset sintético que cumpla con los requisitos del caso 1, los encargos documentados y los patrones observados en los proyectos de referencia:
- StreamView Analytics: limpieza, EDA, variables semánticas, foco en sesgos estructurales y calidad de fuente.
- Telco churn ML: cuidado de train/test, objetivo de negocio, métricas de prioridad y tratamiento de variables.

El objetivo es generar registros coherentes con perfiles de clientes y servicios, pero incluyendo sesgos explícitos y reglas que hagan el caso realista desde la perspectiva de negocio y análisis de riesgo.